# Data Preparation

Takes the cohort from the extraction step, applies the cleaning rules set out in the
methodology, and splits the data into training and test sets. Everything downstream loads
the files written here.

The split is built once, in one place, deliberately. If each generator built its own
split, small differences between them would make the comparison unfair. One shared split
means every method trains on the same records and is judged on the same held-out
patients.

## Setup

In [1]:
%pip install -q pandas pyarrow numpy

## Working folder

Sets the project folder so everything the pipeline writes, the cohort, the synthetic
datasets, the outputs and the figures, persists between sessions rather than sitting on
temporary storage.

The cohort and the synthetic datasets derive from MIMIC-IV, which is credentialed data
under a PhysioNet data use agreement. Keep the folder private, do not share it, and
delete the data once the work is finished.

In [2]:
import os
from pathlib import Path

# Use the shared project folder when one is available, otherwise stay in the
# current directory.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    project_dir = Path("/content/drive/MyDrive/mimic-synthetic-pipeline")
    project_dir.mkdir(parents=True, exist_ok=True)
    os.chdir(project_dir)
    print(f"Working folder: {project_dir}")
except ImportError:
    print("Using the local working folder.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working folder: /content/drive/MyDrive/mimic-synthetic-pipeline


## Cleaning

Two rules, both from the methodology. Missing categorical values become an explicit
Missing category rather than being dropped or imputed, because a value not being recorded
can itself say something about how the patient moved through the hospital. Missing length
of stay is filled with the median.

The identifier columns are dropped here. They were needed to build the cohort and to
split by patient, but they carry no clinical meaning and must not reach a generator,
where reproducing them would be a privacy risk in itself.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

cohort_path = Path("data/cohort.parquet")
if not cohort_path.exists():
    raise FileNotFoundError(
        "data/cohort.parquet not found. Run the extraction step first."
    )

TARGET = "readmitted_30d"
NUMERIC_COLS = ["age_at_admission", "length_of_stay_days"]
CATEGORICAL_COLS = [
    "gender",
    "admission_type",
    "admission_location",
    "insurance",
    "marital_status",
    "race",
    "language",
    "had_icu_stay",
]
MODEL_COLS = NUMERIC_COLS + CATEGORICAL_COLS + [TARGET]

SEED = 42  # fixed random seed so the split is identical every time this runs


def clean(df: pd.DataFrame) -> pd.DataFrame:
    """Select the modelling columns and apply the missing-value rules from Chapter 3."""
    df = df[MODEL_COLS].copy()
    for col in CATEGORICAL_COLS:
        df[col] = df[col].astype("object").fillna("Missing").astype(str)
    df["length_of_stay_days"] = df["length_of_stay_days"].fillna(df["length_of_stay_days"].median()).astype("float64")
    # BigQuery returns integers as pandas nullable Int64, which some generator
    # libraries cannot read. Convert them to plain numpy int64 so every
    # downstream notebook receives standard data types.
    df["age_at_admission"] = df["age_at_admission"].astype("int64")
    df[TARGET] = df[TARGET].astype("int64")
    return df


df = pd.read_parquet(cohort_path)
print(f"Cohort loaded: {len(df):,} admissions from {df['subject_id'].nunique():,} patients")

Cohort loaded: 534,182 admissions from 218,182 patients


## Splitting by patient

An 80/20 train and test split, done at patient level. 45.1% of patients in this cohort
have more than one admission, so shuffling admissions individually would put the same
patient on both sides, and a model would be tested on people it had already seen.
Splitting by patient keeps all of a person's admissions together.

The generators only ever see the training set. The test set is held back to judge whether
models trained on synthetic data work on genuinely unseen patients.

In [4]:
rng = np.random.default_rng(SEED)
patients = np.asarray(df["subject_id"].unique())
rng.shuffle(patients)
n_test = int(len(patients) * 0.2)
test_patients = set(patients[:n_test])

train_df = clean(df[~df["subject_id"].isin(test_patients)].reset_index(drop=True))
test_df = clean(df[df["subject_id"].isin(test_patients)].reset_index(drop=True))

train_df.to_parquet("data/train.parquet", index=False)
test_df.to_parquet("data/test.parquet", index=False)

print(f"Train: {len(train_df):,} admissions | Test: {len(test_df):,} admissions")
print(f"Train readmission rate: {train_df[TARGET].mean():.4f} | Test readmission rate: {test_df[TARGET].mean():.4f}")
print("Saved to data/train.parquet and data/test.parquet")

/tmp/ipykernel_2806/2002319731.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].astype("object").fillna("Missing").astype(str)
/tmp/ipykernel_2806/2002319731.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].astype("object").fillna("Missing").astype(str)


Train: 427,408 admissions | Test: 106,774 admissions
Train readmission rate: 0.2067 | Test readmission rate: 0.2048
Saved to data/train.parquet and data/test.parquet


The two readmission rates should be close to each other and to the overall rate of about
20.6%, which confirms the split did not concentrate readmissions on one side.